# Run survival pipeline locally

Same commands as `bash_scripts/launch_survival.sh` and `array_survival_run.sh`, plus
the upstream preprocessing step that re-applies the sentinel + per-measurement
physiologic-range filters added to `consolidate_dfci_labs.py` (so this notebook
covers end-to-end: raw labs → consolidated → prediction inputs → models).

**Before running:** make sure the active kernel is the `survlatent_ode` conda env (or
activate it inline with `conda run -n survlatent_ode python …`). Edit the `Paths`
cell to point at your data.


## Paths

Defaults are the cluster paths from `bash_scripts/build_prediction_inputs.sh`. Override `PROJECT_ROOT`, `DATA`, `V3_LABELS`, `INPUTS_DIR`, `OUTPUT_DIR` for local copies.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path("/data/gusev/USERS/jpconnor/code/CAIA")
PREP_DIR     = PROJECT_ROOT / "COMPASS" / "data_preprocessing"
SURVIVAL_DIR = PROJECT_ROOT / "COMPASS" / "survival_analysis"

# Upstream (preprocessing) paths — must point at the COMPASS data dir that
# longitudinal_data_processing.py reads/writes.
NEPC_PROJ_PATH        = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/")
CONSOLIDATED_LABS     = NEPC_PROJ_PATH / "consolidated_longitudinal_data.csv"
DATA                  = NEPC_PROJ_PATH / "longitudinal_prediction_data.csv"  # output of step 1
V3_LABELS             = NEPC_PROJ_PATH / "v3_outputs" / "LLM_v3_labels.tsv"

# Downstream (survival) paths
INPUTS_DIR = NEPC_PROJ_PATH / "survival_analysis" / "prediction_inputs"
OUTPUT_DIR = NEPC_PROJ_PATH / "survival_analysis" / "local_runs"

os.chdir(PROJECT_ROOT)
INPUTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# BLAS thread caps (mirror the bash scripts so models don't oversubscribe cores).
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

print("cwd:               ", os.getcwd())
print("prep_dir:          ", PREP_DIR)
print("survival_dir:      ", SURVIVAL_DIR)
print("consolidated_labs: ", CONSOLIDATED_LABS)
print("longitudinal_data: ", DATA)
print("inputs_dir:        ", INPUTS_DIR)
print("output_dir:        ", OUTPUT_DIR)


## 1. Preprocess raw labs (apply sentinel + physiologic-range filters)

Re-runs `longitudinal_data_processing.py`, which internally calls
`consolidate_dfci_labs.consolidate_dfci_labs()`. That call now applies the
9999999.0 sentinel filter (across every lab) and the per-measurement
physiologic-range filter (mirrored from `caia-project-compass`), so any change
to those filters takes effect here.

Watch stdout for:

```
[consolidate_dfci_labs] dropped N rows whose NUMERIC_RESULT matched a known sentinel value (e.g. 9999999.0).
```

A large `N` confirms the testosterone-sentinel bug was real in the raw pulls.
Skip this cell if you've already re-run preprocessing after the most recent
filter change.


In [ ]:
!python {PREP_DIR}/longitudinal_data_processing.py


## 2. Verify the sentinel + range filters fired

Spot-check the consolidated output:

- `Testosterone` standardized max should be ≤ 2000 ng/dL (was 9,999,999 before fix).
- `PSA` standardized max should be ≤ 10,000 ng/mL.
- `out_of_physiologic_range` should appear in `conversion_status` for any
  outliers caught by the second filter.


In [ ]:
import pandas as pd

df = pd.read_csv(CONSOLIDATED_LABS, low_memory=False)
print(f"consolidated rows: {len(df):,}")
print()

# Out-of-range counts by measurement (the per-lab physiologic-range filter)
oor = df.loc[df["conversion_status"] == "out_of_physiologic_range", "collapsed_measurement"]
print("out_of_physiologic_range rows by collapsed_measurement (top 15):")
print(oor.value_counts().head(15).to_string() if not oor.empty else "  (none — either no outliers or filter not active)")
print()

# Direct spot-check of the two labs we care about for the platinum analysis
for lab, bound, unit in [("Testosterone", 2000, "ng/dL"), ("PSA", 10000, "ng/mL")]:
    sub = df.loc[(df["collapsed_measurement"] == lab) & df["numeric_result_standardized"].notna(), "numeric_result_standardized"]
    if sub.empty:
        print(f"  {lab}: no standardized values present")
        continue
    over = (sub > bound).sum()
    print(f"  {lab:13s}: n={len(sub):>6,}  max={sub.max():.2f} {unit}  "
          f"#>{bound}={over}  (should be 0)")


## 3. Build prediction inputs

Wipes the cached `aggregated_landmark{D}.csv` / `longitudinal_landmark{D}.csv`
(they still contain sentinel-contaminated lab features from before the fix) and
rebuilds from the freshly re-consolidated longitudinal data. Required after any
change to `longitudinal_prediction_data.csv` or `build_prediction_inputs.py`.


In [ ]:
# Wipe cached per-landmark inputs so they're rebuilt from the fixed raw data.
import os
for pattern in (
    "aggregated_landmark*.csv",
    "longitudinal_landmark*.csv",
    "longitudinal_landmark*_manifest.json",
    "pre_treatment_lab_long_landmark*.csv",
):
    for p in INPUTS_DIR.glob(pattern):
        p.unlink()
        print(f"  removed {p.name}")
for fname in (
    "canonical_labs_train_val.csv",
    "build_manifest.json",
    "split_assignments.csv",
    "landmark_mrn_availability.csv",
):
    p = INPUTS_DIR / fname
    if p.exists():
        p.unlink()
        print(f"  removed {p.name}")
print("  cache cleared\n")


In [ ]:
!python {SURVIVAL_DIR}/build_prediction_inputs.py \
    --data {DATA} \
    --v3-labels-path {V3_LABELS} \
    --output-dir {INPUTS_DIR} \
    --landmark-days 0 90 \
    --time-unit-days 7 \
    --test-frac 0.20 \
    --val-frac 0.20 \
    --min-patient-coverage 0.20


## 4. Cohort diagnostics post-fix

Spot-checks the rebuilt aggregated CSV for the testosterone fix. The platinum-
positive `Testo max` median should be in the 100–1500 ng/dL range now (was
9,999,999 pre-fix). PSA medians should be unchanged since PSA was already
sentinel-filtered upstream by `compile_prostate_data.py`.


In [ ]:
import pandas as pd, numpy as np

for lm in (0, 90):
    agg_path = INPUTS_DIR / f"aggregated_landmark{lm}.csv"
    if not agg_path.exists():
        print(f"  landmark +{lm}d: aggregated CSV not found, skipping")
        continue
    agg = pd.read_csv(agg_path)

    def find_col(substr, stat):
        return next(
            (c for c in agg.columns if substr.lower() in c.lower() and c.endswith(f"__{stat}")),
            None,
        )

    n_plat = int(agg["PLATINUM"].sum())
    n_death = int(agg["DEATH"].sum())
    print(f"=== landmark +{lm}d  |  n_total={len(agg):,}  n_PLATINUM={n_plat}  n_DEATH={n_death} ===")
    for lab_substr in ("Testosterone", "PSA", "Prostate specific Ag"):
        for stat in ("mean", "last", "max", "min"):
            col = find_col(lab_substr, stat)
            if col is None:
                continue
            for ev in (0, 1):
                sub = agg.loc[agg["PLATINUM"] == ev, col].dropna()
                if sub.empty:
                    continue
                print(
                    f"  {lab_substr:>22s} {stat:5s}  PLAT={ev}:  median={sub.median():>10.2f}  "
                    f"max={sub.max():>12.2f}  n={len(sub):>5}"
                )
            break  # only report once per lab (use whichever variant matched)
    print()


## 5. Run all platinum tasks at landmarks 0 and 90

Runs cox + xgboost + deephit (PLATINUM and competing configs) at both landmarks,
plus the PGS-adjusted Cox sweep for Testosterone + PSA. Skips any task whose
metrics CSV already exists at the expected output path. Set `FORCE_RERUN = True`
to override and rerun everything (do this after a preprocessing-layer fix).


In [ ]:
import subprocess
import time

N_FOLDS = 5
FORCE_RERUN = False

# (model, landmark, config_dir, metrics_filename)
# config_dir is the subfolder under landmark_{D}/ where outputs are written.
TASKS = [
    ("cox",     0,  "both",      "cox_agg_multivariable_metrics.csv"),
    ("cox",     90, "both",      "cox_agg_multivariable_metrics.csv"),
    ("xgboost", 0,  "both",      "landmark_xgboost_metrics.csv"),
    ("xgboost", 90, "both",      "landmark_xgboost_metrics.csv"),
    ("deephit", 0,  "PLATINUM",  "dynamic_deephit_metrics_PLATINUM.csv"),
    ("deephit", 90, "PLATINUM",  "dynamic_deephit_metrics_PLATINUM.csv"),
    ("deephit", 0,  "competing", "dynamic_deephit_metrics_competing.csv"),
    ("deephit", 90, "competing", "dynamic_deephit_metrics_competing.csv"),
]


def build_command(model, landmark, config_dir, row_output_dir):
    if model == "cox":
        return [
            "python", str(SURVIVAL_DIR / "cox_aggregated.py"),
            "--inputs-dir", str(INPUTS_DIR),
            "--output-dir", str(row_output_dir),
            "--landmark-days", str(landmark),
            "--analysis", config_dir,           # "both" runs univariate + multivariable
            "--endpoints", "platinum", "death",
            "--n-folds", str(N_FOLDS),
        ]
    if model == "xgboost":
        return [
            "python", str(SURVIVAL_DIR / "landmark_xgboost.py"),
            "--inputs-dir", str(INPUTS_DIR),
            "--output-dir", str(row_output_dir),
            "--landmark-days", str(landmark),
            "--endpoints", "platinum", "death",
            "--n-folds", str(N_FOLDS),
        ]
    if model == "deephit":
        return [
            "python", str(SURVIVAL_DIR / "dynamic_deephit.py"),
            "--inputs-dir", str(INPUTS_DIR),
            "--output-dir", str(row_output_dir),
            "--landmark-day", str(landmark),
            "--config", config_dir,
            "--n-folds", str(N_FOLDS),
        ]
    raise ValueError(f"Unknown model: {model}")


summary = []
for model, landmark, config_dir, metrics_filename in TASKS:
    row_output_dir = OUTPUT_DIR / model / f"landmark_{landmark}" / config_dir
    metrics_path = row_output_dir / metrics_filename
    tag = f"{model:8s}  landmark_{landmark:<2}  {config_dir}"

    if metrics_path.exists() and not FORCE_RERUN:
        print(f"[skip] {tag}  ->  {metrics_path.relative_to(OUTPUT_DIR)} exists")
        summary.append((tag, "skipped", 0.0))
        continue

    row_output_dir.mkdir(parents=True, exist_ok=True)
    cmd = build_command(model, landmark, config_dir, row_output_dir)
    print(f"[run ] {tag}")
    print("       " + " ".join(cmd))
    t0 = time.time()
    rc = subprocess.call(cmd)
    elapsed = time.time() - t0
    status = "ok" if rc == 0 else f"FAILED (rc={rc})"
    print(f"[done] {tag}  -> {status}  ({elapsed/60:.1f} min)\n")
    summary.append((tag, status, elapsed))

print("\n=== run summary ===")
for tag, status, elapsed in summary:
    print(f"  {tag}  {status:>20s}  {elapsed/60:6.1f} min")

# Optional: PGS-adjusted Cox sweep for Testosterone + PSA. Writes
# OUTPUT_DIR/cox/landmark_{D}/both/cox_pgs_adjusted.csv per landmark.
# This is the analysis where the sentinel/range fix matters most — the
# testosterone HR direction should flip once the 9999999 values are gone.
RUN_PGS_ADJUSTED = True
if RUN_PGS_ADJUSTED:
    pgs_out_dir = OUTPUT_DIR / "cox"
    pgs_out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        "python", str(SURVIVAL_DIR / "cox_pgs_adjusted.py"),
        "--aggregated-csv-pattern", str(INPUTS_DIR / "aggregated_landmark{landmark}.csv"),
        "--output-dir", str(pgs_out_dir),
        "--landmarks", "0", "90",
        "--endpoints", "platinum", "death",
        "--target-labs", "Testosterone", "PSA",
        "--feature-stat", "mean",
    ]
    print("[run ] cox_pgs_adjusted")
    print("       " + " ".join(cmd))
    t0 = time.time()
    rc = subprocess.call(cmd)
    print(f"[done] cox_pgs_adjusted -> rc={rc}  ({(time.time()-t0)/60:.1f} min)")


## 6. Inspect outputs

Pulls the headline metrics row from each task's metrics CSV into one comparison table.


In [ ]:
import pandas as pd

rows = []
for model, landmark, config_dir, metrics_filename in TASKS:
    metrics_path = OUTPUT_DIR / model / f"landmark_{landmark}" / config_dir / metrics_filename
    if not metrics_path.exists():
        rows.append({
            "model": model, "landmark": landmark, "config": config_dir,
            "endpoint": "-", "n_test": None, "n_test_events": None,
            "c_index": None, "mean_auc_t": None, "integrated_brier": None,
            "status": "missing",
        })
        continue
    df = pd.read_csv(metrics_path)

    if model == "cox":
        platinum = df[df["endpoint"] == "platinum"].iloc[0]
        rows.append({
            "model": model, "landmark": landmark, "config": config_dir,
            "endpoint": "platinum",
            "n_test": int(platinum["n_test"]),
            "n_test_events": int(platinum["n_events_test"]),
            "c_index": float(platinum["test_c_index"]),
            "mean_auc_t": float(platinum["test_mean_auc_t"]),
            "integrated_brier": float(platinum["test_integrated_brier"]),
            "status": "ok",
        })
    elif model == "xgboost":
        platinum = df[df["endpoint"] == "platinum"].iloc[0]
        rows.append({
            "model": model, "landmark": landmark, "config": config_dir,
            "endpoint": "platinum",
            "n_test": int(platinum["n_test"]),
            "n_test_events": int(platinum["n_test_events"]),
            "c_index": float(platinum["c_index"]),
            "mean_auc_t": float(platinum["mean_auc_t"]),
            "integrated_brier": float(platinum["integrated_brier"]),
            "status": "ok",
        })
    elif model == "deephit":
        for _, r in df.iterrows():
            rows.append({
                "model": model, "landmark": landmark, "config": config_dir,
                "endpoint": str(r["event"]),
                "n_test": int(r["n_test"]),
                "n_test_events": int(r["n_test_events"]),
                "c_index": float(r["c_index"]),
                "mean_auc_t": float(r["mean_auc_t"]),
                "integrated_brier": float(r["integrated_brier"]),
                "status": "ok",
            })

summary_df = pd.DataFrame(rows)
summary_df = summary_df.sort_values(["endpoint", "landmark", "model", "config"]).reset_index(drop=True)
summary_df